# 02 · Fine-Tune a Model (LoRA)  *(Instructor · GPU)*

**Approach 2 of 3.** Instead of starting from zero, we take a model that **already understands
language** and gently **adapt** it to answer in our style/format, using our corpus.

We use **LoRA** (Low-Rank Adaptation) — it trains a tiny set of extra weights (an "adapter") instead
of the whole model. That's why fine-tuning fits on one GPU and the result we share is only a few MB.

> **What fine-tuning is good at:** new **behavior, tone, and format.** It's *less* reliable for
> injecting brand-new facts — for that, you'll want RAG (Notebook 03).

In [ ]:
import json, torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from datasets import Dataset

ARTIFACTS = Path("artifacts")
BASE_MODEL = "Qwen/Qwen3-1.7B"
device = "cuda" if torch.cuda.is_available() else "cpu"
corpus = json.load(open(ARTIFACTS / "corpus.json"))
print("Device:", device, "| corpus entries:", len(corpus))

### Build a small instruction dataset from the corpus
Fine-tuning learns from **examples of the behavior we want** — here, concise Q&A in a consistent style.

In [ ]:
def example(entry):
    return {"messages": [
        {"role": "user", "content": f"Define '{entry['title']}' clearly and concisely."},
        {"role": "assistant", "content": entry["text"]},
    ]}

train_ds = Dataset.from_list([example(e) for e in corpus])
print("Training examples:", len(train_ds))
print(train_ds[0]["messages"])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def to_text(ex):
    # Render each chat example with the model's template into one training string.
    return {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False,
                                                  add_generation_prompt=False)}
train_text = train_ds.map(to_text, remove_columns=["messages"])
print(train_text[0]["text"][:300])

In [ ]:
from trl import SFTTrainer, SFTConfig

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype="auto").to(device)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                  task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])

sft_args = SFTConfig(output_dir=str(ARTIFACTS / "_lora_train"),
                     num_train_epochs=10, per_device_train_batch_size=2,
                     learning_rate=2e-4, logging_steps=5, report_to=[],
                     dataset_text_field="text", max_seq_length=512, save_strategy="no")
trainer = SFTTrainer(model=model, args=sft_args, train_dataset=train_text, peft_config=lora)
trainer.train()

trainer.model.save_pretrained(ARTIFACTS / "lora_adapter")
tokenizer.save_pretrained(ARTIFACTS / "lora_adapter")
print("Saved LoRA adapter (small!) to", ARTIFACTS / "lora_adapter")

### Optional: share the adapter on Hugging Face
The adapter is only a few MB — perfect for sharing. Set `HF_REPO` in Notebook 00 and log in first
(`huggingface-cli login`). Skipped if `HF_REPO` is None.

In [ ]:
HF_REPO = None  # e.g. "your-org/ttr-demo-lora"
if HF_REPO:
    trainer.model.push_to_hub(HF_REPO)
    tokenizer.push_to_hub(HF_REPO)
    print("Pushed adapter to", HF_REPO)
else:
    print("Skipped HF upload (HF_REPO is None). Share the artifacts/ folder instead.")

**Takeaway:** fine-tuning reused everything the base model already knew about language and just
taught it our *style*. It cost a fraction of training from scratch and the shareable result is tiny.
But notice we taught **behavior**, not new facts — for the private Redlake facts, we need RAG.